In [1]:
%load_ext autoreload
%autoreload 2


In [11]:
import numpy as np
import matplotlib.pyplot as plt
import qiskit_metal as metal
from qiskit_metal import Dict, designs, MetalGUI
from qiskit_metal.qlibrary.tlines.straight_path import RouteStraight
from qiskit_metal.qlibrary.terminations.open_to_ground import OpenToGround
import numpy as np
from collections import OrderedDict
from qiskit_metal.qlibrary.tlines.anchored_path import RouteAnchors
from qiskit_metal.qlibrary.tlines.framed_path import RouteFramed
from qiskit_metal.qlibrary.terminations.launchpad_wb import LaunchpadWirebond
from qiskit_metal.qlibrary.lumped.cap_n_interdigital import CapNInterdigital

In [3]:
design = designs.DesignPlanar(overwrite_enabled=True)

In [22]:
gui = MetalGUI(design)
design.variables['cpw_width']='10um'
design.variables['cpw_gap']='6um'

# Calculos das capacitancias

In [20]:
l = 1000
d = np.linspace(0, 1000, 100)

a0 = 1.62129e-7
a1 = 0.100845
b0 = 0.0592123
b1 = -2.29999e-6
m0 = 0.000600864
m1 = -2.1275e-6
c0 = -0.201206
c1 = 0.00156082
A = a0 + a1*l
B = b0 + b1*l
M = m0 + m1*l
C = c0 + c1*l

C = A*np.exp(-B*d) + M*d + C

plt.plot(d, C)

In [52]:
def Cap(d, l):
    """d (um) - distancia do gap entre os dois metais
        l (um) - comprimento dos metais que ficam paralelos"""
    a0 = 1.62129e-7
    a1 = 0.100845
    b0 = 0.0592123
    b1 = -2.29999e-6
    m0 = 0.000600864
    m1 = -2.1275e-6
    c0 = -0.201206
    c1 = 0.00156082
    A = a0 + a1*l
    B = b0 + b1*l
    M = m0 + m1*l
    C = c0 + c1*l
    
    C = A*np.exp(-B*d) + M*d + C
    return C

Cap(800, 4000)

-0.2852347999999987

# Um ressonador

In [68]:
def ressonador_cima(name, connection1, connection2):
    """
    name - string com o nome do ressonador
    connection = [nome, pos_x, pos_y], todos nas forma de string"""

    OpenToGround(design, connection1[0], options=Dict(
        pos_x=connection1[1],
        pos_y=connection1[2],
        orientation='90',
        #width='10um',
        #gap='6um',
        #termination_gap='12um'
    ))

    OpenToGround(design, connection2[0], options=Dict(
        pos_x=connection2[1],
        pos_y=connection2[2],
        orientation='90',
        #width='10um',
        #gap='6um',
        #termination_gap='12um'
    ))

    return RouteFramed(design, name, options=Dict(
        pin_inputs = Dict(
            start_pin = Dict(
                component = connection1[0],
                pin = 'open'
            ),
            end_pin = Dict(
                component = connection2[0],
                pin = 'open'
            )
        ),
        #total_length='10mm',
        lead=Dict(start_straight='4.15mm', end_straight='4.15mm'),
        fillet='300um',
        hfss_wire_bonds = True
    ))

In [69]:
def ressonador_baixo(name, connection1, connection2):
    """
    name - string com o nome do ressonador
    connection = [nome, pos_x, pos_y], todos nas forma de string"""

    OpenToGround(design, connection1[0], options=Dict(
        pos_x=connection1[1],
        pos_y=connection1[2],
        orientation='-90',
        #width='10um',
        #gap='6um',
        #termination_gap='12um'
    ))

    OpenToGround(design, connection2[0], options=Dict(
        pos_x=connection2[1],
        pos_y=connection2[2],
        orientation='-90',
        #width='10um',
        #gap='6um',
        #termination_gap='12um'
    ))

    return RouteFramed(design, name, options=Dict(
        pin_inputs = Dict(
            start_pin = Dict(
                component = connection1[0],
                pin = 'open'
            ),
            end_pin = Dict(
                component = connection2[0],
                pin = 'open'
            )
        ),
        total_length='10mm',
        lead=Dict(start_straight='4.15mm', end_straight='4.15mm'),
        fillet='300um',
        hfss_wire_bonds = True
    ))

In [70]:
res1 = ressonador_cima('RES1', ['otg1', '0mm', '0mm'], ['otg2', '2mm', '0mm'])
res2 = ressonador_baixo('RES2', ['otg3', '-1.4mm', '-3mm'], ['otg4', '0.6mm', '-3mm'])

gui.rebuild()
gui.autoscale()

In [55]:
OpenToGround(design, 'OTG5', options=Dict(
    pos_x='0mm',
    pos_y='0mm',
    orientation='90',
    #width='10um',
    #gap='6um',
    #termination_gap='12um'
))

name:    OTG5
class:   OpenToGround          
options: 
  'pos_x'             : '0mm',                        
  'pos_y'             : '0mm',                        
  'orientation'       : '90',                         
  'chip'              : 'main',                       
  'layer'             : '1',                          
  'width'             : '10um',                       
  'gap'               : '6um',                        
  'termination_gap'   : '6um',                        
module:  qiskit_metal.qlibrary.terminations.open_to_ground
id:      63